# Limpieza y enriquecimiento ENIGH 2018-2024

Notebook de la segunda revisión: valida el catálogo geográfico, construye un catálogo municipal, prueba el merge con `ubica_geo`, homologa tipos técnicos y deja auditada la falta de catálogos para decodificar variables categóricas.


## Orden de trabajo

1. Geografía: inspección, granularidad, catálogo municipal, cobertura y merge `m:1`.
2. Tipos: metadata `C/N`, corrección de artefactos `1.0` en 2024 y validación posterior.
3. Categóricas: inventario y bloqueo explícito cuando faltan catálogos de códigos.


In [1]:
from pathlib import Path
import pandas as pd

def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / 'README.md').exists() and (candidate / 'data').exists():
            return candidate
    raise FileNotFoundError('No pude localizar la raíz del proyecto.')

ROOT = find_project_root()
REV2 = ROOT / 'data' / 'interim' / 'revision_2'
required_files = [
    'validacion_archivos_geografia.csv',
    'validacion_cobertura_geografia.csv',
    'validacion_merge_geografia.csv',
    'resumen_observaciones_por_municipio.csv',
    'validacion_tipos.csv',
    'inventario_categoricas.csv',
    'resumen_revision_2.json',
]
missing = [name for name in required_files if not (REV2 / name).exists()]
if missing:
    raise FileNotFoundError(f'Faltan outputs esperados de revisión 2: {missing}')

print(f'Raíz del proyecto: {ROOT}')
print('Revision 2 outputs:')
for path in sorted(REV2.glob('*')):
    print(path.name)


Raíz del proyecto: c:\Users\lucia\OneDrive\Escritorio\Fer\inegi-income-modeling
Revision 2 outputs:
archivos_revision_2.csv
catalogo_municipal.csv
concentradohogar_limpia_enriquecida_2018_2024.csv.gz
hogares_limpia_enriquecida_2018_2024.csv.gz
ingresos_limpia_enriquecida_2018_2024.csv.gz
inventario_categoricas.csv
observaciones_por_municipio.csv
poblacion_limpia_enriquecida_2018_2024.csv.gz
resumen_observaciones_por_municipio.csv
resumen_revision_2.json
trabajos_limpia_enriquecida_2018_2024.csv.gz
validacion_archivos_geografia.csv
validacion_catalogo_municipal.csv
validacion_cobertura_geografia.csv
validacion_merge_geografia.csv
validacion_tipos.csv
viviendas_limpia_enriquecida_2018_2024.csv.gz


In [2]:
pd.read_csv(REV2 / 'validacion_archivos_geografia.csv')


,archivo,formato,filas,columnas,nivel_aparente,posibles_llaves,nota
0,AGEEML_20268131444460_utf.csv,CSV,360473,21.0,localidad,Mapa; Clave de AGEE + Clave de AGEM + Clave Lo...,La variante *_utf muestra mojibake en acentos;...
1,AGEEML_20268131456832_utf.txt,TXT,360473,21.0,localidad,Mapa; Clave de AGEE + Clave de AGEM + Clave Lo...,"Mismo contenido tabular que CSV, separado por ..."
2,AGEEML_20268131556800.dbf,DBF,360473,20.0,localidad,Campos truncados de AGEE/AGEM/Mapa,Los nombres de campos se truncan a 10 caracteres.
3,AGEEML_2026813173099.xlsx,XLSX,360477,20.0,localidad,Mapa; Clave de AGEE + Clave de AGEM + Clave Lo...,Incluye filas de título/fecha antes del encabe...
4,AGEEML_20268131742140.csv,CSV,360473,21.0,localidad,Mapa; Clave de AGEE + Clave de AGEM + Clave Lo...,La variante *_utf muestra mojibake en acentos;...
5,AGEEML_20268131754722.txt,TXT,360473,21.0,localidad,Mapa; Clave de AGEE + Clave de AGEM + Clave Lo...,"Mismo contenido tabular que CSV, separado por ..."
6,AGEEML_202681494756.kml,KML,360473,NaN,localidad/geometría,Placemark/Mapa,No se usa para merge municipal ni coordenadas.


In [3]:
pd.read_csv(REV2 / 'validacion_cobertura_geografia.csv')


,tabla,anio,registros,ubica_geo_unicos,pct_validos,pct_match,pct_sin_match,claves_sin_match
0,concentradohogar,2018,74647,996,100.0,100.0,0.0,NaN
1,concentradohogar,2020,89006,1090,100.0,100.0,0.0,NaN
2,concentradohogar,2022,90102,1132,100.0,100.0,0.0,NaN
3,concentradohogar,2024,91414,1112,100.0,100.0,0.0,NaN
4,viviendas,2018,73405,996,100.0,100.0,0.0,NaN
5,viviendas,2020,87754,1090,100.0,100.0,0.0,NaN
6,viviendas,2022,88823,1132,100.0,100.0,0.0,NaN
7,viviendas,2024,90324,1112,100.0,100.0,0.0,NaN


In [4]:
pd.read_csv(REV2 / 'validacion_merge_geografia.csv')


,tabla,filas_antes,filas_despues,diferencia,estado
0,concentradohogar,345169,345169,0,OK
1,hogares,345169,345169,0,Sin ubica_geo; no aplica merge geografico
2,ingresos,1532144,1532144,0,Sin ubica_geo; no aplica merge geografico
3,poblacion,1203231,1203231,0,Sin ubica_geo; no aplica merge geografico
4,trabajos,634140,634140,0,Sin ubica_geo; no aplica merge geografico
5,viviendas,340306,340306,0,OK


In [5]:
pd.read_csv(REV2 / 'resumen_observaciones_por_municipio.csv')


,tabla,anio,municipios_observados,min,p10,p25,mediana,p75,p90,max,lt_5,lt_10,lt_20,lt_30,lt_50,lt_100
0,concentradohogar,2018,996,4,19.0,22.0,34.0,69.25,146.5,1384,1,21,124,485,655,823
1,concentradohogar,2020,1090,4,18.0,21.0,39.0,70.00,171.1,1500,3,21,180,471,699,904
2,concentradohogar,2022,1132,3,18.0,20.0,35.5,66.25,169.5,1544,3,25,212,534,740,924
3,concentradohogar,2024,1112,2,19.0,22.0,40.0,79.00,177.9,1689,8,41,126,525,719,904
4,viviendas,2018,996,4,19.0,22.0,34.0,68.00,144.0,1358,1,22,131,487,662,828
5,viviendas,2020,1090,4,18.0,21.0,39.0,69.75,168.1,1477,3,21,192,473,705,905
6,viviendas,2022,1132,3,18.0,20.0,35.0,65.00,166.6,1525,3,26,228,534,741,927
7,viviendas,2024,1112,2,19.0,22.0,40.0,79.00,176.9,1670,8,42,132,525,720,906


In [6]:
tipos = pd.read_csv(REV2 / 'validacion_tipos.csv')
columnas_tipos = [
    'tabla', 'variable', 'tipo_2018', 'tipo_2020', 'tipo_2022', 'tipo_2024',
    'tipo_metadata', 'tipo_conceptual', 'regla_final',
    'artefactos_decimal_antes', 'artefactos_decimal_2024', 'artefactos_decimal_despues',
    'problema', 'valores_no_numericos',
]
tipos[columnas_tipos].head(30)


,tabla,variable,tipo_2018,tipo_2020,tipo_2022,tipo_2024,tipo_metadata,tipo_conceptual,regla_final,artefactos_decimal_antes,artefactos_decimal_2024,artefactos_decimal_despues,problema,valores_no_numericos
0,concentradohogar,anio,N,N,N,N,N,fecha/levantamiento,Int64,0,0,0,NaN,NaN
1,concentradohogar,folioviv,C (10),C (10),C (10),C (10),C,identificador,string_codigo_zfill_10,0,0,0,NaN,NaN
2,concentradohogar,foliohog,C (1),C (1),C (1),C (1),C,identificador,string_codigo_zfill_1,0,0,0,NaN,NaN
3,concentradohogar,ubica_geo,C (5),C (5),C (5),C (5),C,llave geografica,string_codigo_zfill_5,0,0,0,NaN,NaN
4,concentradohogar,tam_loc,C (1),C (1),C (1),C (1),C,categorica codificada,string_codigo_zfill_1,0,0,0,NaN,NaN
5,concentradohogar,est_socio,C (1),C (1),C (1),C (1),C,categorica codificada,string_codigo_zfill_1,0,0,0,NaN,NaN
6,concentradohogar,est_dis,C (7),C (3),C (3),C (3),C,diseno muestral,string_codigo,0,0,0,NaN,NaN
7,concentradohogar,upm,C (5),C (7),C (7),C (7),C,diseno muestral,string_codigo,0,0,0,NaN,NaN
8,concentradohogar,factor,N (5),N (5),N (5),N (5),N,numerica,Int64,0,0,0,NaN,NaN
9,concentradohogar,clase_hog,C (1),C (1),C (1),C (1),C,categorica codificada,string_codigo_zfill_1,0,0,0,NaN,NaN


In [7]:
pd.read_csv(REV2 / 'inventario_categoricas.csv').head(30)


,tabla,variable,tipo,doc_2018,doc_2020,doc_2022,doc_2024,codificacion_estable,grupo
0,concentradohogar,tam_loc,categorica codificada,Si,Si,Si,Si,No validada: faltan etiquetas/codigos,C. Documentacion faltante
1,concentradohogar,est_socio,categorica codificada,Si,Si,Si,Si,No validada: faltan etiquetas/codigos,C. Documentacion faltante
2,concentradohogar,clase_hog,categorica codificada,Si,Si,Si,Si,No validada: faltan etiquetas/codigos,C. Documentacion faltante
3,concentradohogar,sexo_jefe,categorica codificada,Si,Si,Si,Si,No validada: faltan etiquetas/codigos,C. Documentacion faltante
4,concentradohogar,educa_jefe,categorica codificada,Si,Si,Si,Si,No validada: faltan etiquetas/codigos,C. Documentacion faltante
5,hogares,acc_alim1,categorica codificada,Si,Si,Si,Si,No validada: faltan etiquetas/codigos,C. Documentacion faltante
6,hogares,acc_alim2,categorica codificada,Si,Si,Si,Si,No validada: faltan etiquetas/codigos,C. Documentacion faltante
7,hogares,acc_alim3,categorica codificada,Si,Si,Si,Si,No validada: faltan etiquetas/codigos,C. Documentacion faltante
8,hogares,acc_alim4,categorica codificada,Si,Si,Si,Si,No validada: faltan etiquetas/codigos,C. Documentacion faltante
9,hogares,acc_alim5,categorica codificada,Si,Si,Si,Si,No validada: faltan etiquetas/codigos,C. Documentacion faltante


Nota metodológica: en esta revisión 2 no se agregaron columnas `*_desc` para variables ENIGH como `sexo`, `tam_loc` o `tipo_viv`; el objetivo fue dejar limpia la estructura, geografía y tipos. La decodificación con catálogos oficiales queda documentada y ejecutada después en `03_decodificacion_categoricas.ipynb`, para no mezclar la limpieza técnica con la interpretación de códigos.
